# Interpretability-Guided Prompt Engineering

Use mechanistic interpretability to understand prompts: token importance mapping, attention steering analysis, critical context positions, prompt sensitivity, and prompt comparison.

## Why This Matters

This module provides tools for analyzing model internals. Understanding the internal mechanisms of transformer models is the core goal of mechanistic interpretability research — enabling us to move from treating models as black boxes to understanding the algorithms they implement.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prompt_engineering_tools import (
    token_importance_map,
    attention_steering_analysis,
    critical_context_positions,
    prompt_sensitivity_map,
    prompt_comparison,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
# Token importance for the last position's prediction
result = token_importance_map(model, tokens)
print(f'Most important: pos {result["most_important"]}')
print(f'Least important: pos {result["least_important"]}')
print('Ranked positions:')
for pos, score in result['ranked_positions']:
    print(f'  Pos {pos}: importance={score:.4f}')

In [ ]:
# Attention steering analysis
result = attention_steering_analysis(model, tokens)
print(f'Concentration score: {result["concentration_score"]:.4f}')
print('Top steering heads:')
for h in result['steering_heads'][:5]:
    print(f'  L{h["layer"]}H{h["head"]}: attn={h["attention_to_source"]:.4f}, '
          f'peak_pos={h["peak_position"]}, entropy={h["entropy"]:.3f}')

In [ ]:
# Critical context positions
result = critical_context_positions(model, tokens, top_k=5)
print('Critical positions:')
for pos, score in result['critical_positions']:
    print(f'  Pos {pos}: combined_score={score:.4f}')

In [ ]:
# Prompt sensitivity (leave-one-out)
result = prompt_sensitivity_map(model, tokens)
print(f'Mean sensitivity: {result["mean_sensitivity"]:.4f}')
print('Most sensitive positions:')
for pos, score in result['most_sensitive_positions']:
    print(f'  Pos {pos}: sensitivity={score:.4f}')

In [ ]:
# Compare two prompts
tokens_a = tokens
tokens_b = jnp.array([50, 65, 80, 95, 10, 25, 40])
result = prompt_comparison(model, tokens_a, tokens_b)
print(f'Logit diff: {result["logit_diff"]:.4f}')
print(f'Most different layer: {result["most_different_layer"]}')
for layer in result['residual_diff_per_layer']:
    print(f'  Layer {layer["layer"]}: residual_diff={layer["diff"]:.4f}')